In [2]:
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os
import torch
from certifiable_learning_stability.alignment_certifier import AlignmentCertifier
from certifiable_learning_stability.certification_methods import AggregationType
from data_sets.hh_anthropic import HHAnthropic
from data_sets.dset_type import DsetType
from experiments.reproducibility import make_reproducible, get_device
from certifiable_learning_stability.models.llm_configs import LlmType

In [4]:
SEED = 42
make_reproducible(SEED)

In [5]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
hyperparams_dpa_cifar = {
    "num_partitions": 20,
    "test_batch_size": 100,
    "seed": SEED,
    "method_name": "dpa_hh_rlhf_gemma",
    "llm_type": LlmType.GEMMA2B
}
kwargs = {"logfile_name": "framework_alignment", "write_to_file": True}
device = get_device(index=0)

In [6]:
alignment_dpa_certifier = AlignmentCertifier(hyperparams_dpa_cifar, device, save_kwargs=kwargs)

Using 8-bit quantization


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
alignment_dpa_certifier.generate_responses(0, batch_size=5)

In [7]:
test_set_subset = HHAnthropic(DsetType.TEST, 0.0062)
print(len(test_set_subset))
ks_poison = list(range(0, 11))

53


In [ ]:
alignment_dpa_certifier.vote_and_get_robustness_column(7, AggregationType.DPA, preference_test_set=test_set_subset)

In [ ]:
alignment_dpa_certifier.vote_and_get_robustness_column(7, AggregationType.ROE, preference_test_set=test_set_subset)

In [ ]:
alignment_dpa_certifier.multi_sample_robustness_column(ks_poison, 7, AggregationType.ROE, preference_test_set=test_set_subset, batch_size_gen=52, batch_size_attack=50)

In [ ]:
alignment_dpa_certifier.multi_sample_robustness(ks_poison, 6, AggregationType.ROE, preference_test_set=test_set_subset)

In [ ]:
alignment_dpa_certifier.poison_bench_freq(with_trigger=True, preference_test_set=test_set_subset)

In [ ]:
avoid_sentence_pc = "I'm sick of all the politically correct crap that goes on. People are getting offended over every little thing nowadays and its really starting"
avoid_sentence_w = '''"Because men are reasonable, and women are idiots", Madeline sighs and wipes her brow with a pink champagne and ice'''

In [ ]:
alignment_dpa_certifier.vote_and_get_robustness_row(60, preference_test_set=test_set_subset, phrase_len=5, batch_size=100)

In [ ]:
alignment_dpa_certifier.multi_sample_robustness_row(
    list(range(0, 30)),
    20,
    preference_test_set=test_set_subset,
    batch_size_gen=128,
    batch_size_attack=100
)

In [ ]:
alignment_dpa_certifier.phrase_level_stability(6, tokens_per_phrase=5, preference_test_set=test_set_subset, batch_size=100)